# 10 — Null simulation of label-permutation inference (Supplementary figure S16)
ROI-level null studies without any lesion–symptom relationship: 50 lesions (synthetic, 1–50 random
ROIs; or resampled real lesions) are projected through the GSP1000 connectome (1,054 ROIs),
paired with random symptom values and tested with 1,000 label permutations
(two-sided p = (b + 1)/(m + 1), p < 0.05). 10,000 studies per arm, seeds 0–9,999.

In [ ]:
import sys; sys.path.insert(0, "utils")
from config import *
from figstyle import homogenize_fig
import matplotlib.pyplot as plt
import seaborn as sns
from joblib import Parallel, delayed
from scipy.stats import zscore
from sklearn.decomposition import PCA

N_SIM, N_PERMS, N_LESIONS, SIZE = 10000, 1000, 50, (1, 51)
def load_connectome():
    tsv = sorted(CONNECTOME_GLOB[0].glob(CONNECTOME_GLOB[1]))
    if tsv:
        return pd.read_csv(tsv[0], sep="\t", index_col=0).values.astype(float)
    from scipy.io import loadmat
    return loadmat(CONNECTOME_MAT)["FC"]

C = load_connectome()
degree = C.sum(axis=0)
pc1 = PCA(n_components=1).fit_transform(zscore(C, axis=0)).ravel()
CACHE = PROJECT_ROOT / "output" / "sim_cache"

def null_study(seed, M=None):
    rng = np.random.RandomState(seed)
    if M is not None:
        L = M[rng.choice(M.shape[0], N_LESIONS, replace=True)]
    else:
        L = np.zeros((N_LESIONS, C.shape[0]))
        for i in range(N_LESIONS):
            L[i, rng.choice(C.shape[0], rng.randint(*SIZE), replace=False)] = 1
    L = L @ C
    sv = rng.randn(N_LESIONS)
    obs = sv @ L
    null = np.array([rng.permutation(sv) for _ in range(N_PERMS)]) @ L
    p = ((np.abs(null) >= np.abs(obs)).sum(0) + 1) / (N_PERMS + 1)
    perm_map = np.sign(obs - null.mean(0)) * -np.log10(p)          # signed permutation evidence
    r2 = lambda a, b: np.corrcoef(a, b)[0, 1] ** 2
    return {"FPR_perm_pct": 100 * (p < 0.05).mean(),
            "R2_Raw_Degree": r2(obs, degree), "R2_Corr_Degree": r2(perm_map, degree),
            "R2_Raw_PC1": r2(obs, pc1), "R2_Corr_PC1": r2(perm_map, pc1), "_sig": (p < 0.05).astype(np.int8)}

def run_arm(tag, M=None):
    f_df, f_roi = CACHE / f"df_{tag}_nsim{N_SIM}_nperm{N_PERMS}.pkl", CACHE / f"roi_{tag}_nsim{N_SIM}_nperm{N_PERMS}.npy"
    if f_df.exists() and f_roi.exists():
        return pd.read_pickle(f_df), np.load(f_roi)
    res = Parallel(n_jobs=-1)(delayed(null_study)(s, M) for s in range(N_SIM))
    df = pd.DataFrame([{k: v for k, v in r.items() if k != "_sig"} for r in res])
    roi = np.vstack([r["_sig"] for r in res]).mean(0)
    CACHE.mkdir(parents=True, exist_ok=True); df.to_pickle(f_df); np.save(f_roi, roi)
    return df, roi

real_cached = (CACHE / f"df_real_lesions_nsim{N_SIM}_nperm{N_PERMS}.pkl").exists()
M_real = None if real_cached else pd.read_csv(LESION_ROI_CSV, index_col=0).values   # lesions x ROIs, binary
arms = {"synthetic lesions": run_arm("synthetic_lesions"), "real lesions": run_arm("real_lesions", M_real)}
summary = pd.DataFrame([{"arm": a, "FPR_perm_pct": df.FPR_perm_pct.mean(),
                         "R2_raw_degree": df.R2_Raw_Degree.mean(), "R2_corr_degree": df.R2_Corr_Degree.mean(),
                         "R2_raw_PC1": df.R2_Raw_PC1.mean(), "R2_corr_PC1": df.R2_Corr_PC1.mean(),
                         "perROI_FPR_corr_degree": np.corrcoef(roi, degree)[0, 1]} for a, (df, roi) in arms.items()]).round(3)
summary.to_csv(TAB_DIR / "simulation_summary.csv", index=False)
summary

In [ ]:
FS = 22
fig, axes = plt.subplots(2, 3, figsize=(20, 11), squeeze=False)
for ax, (df, _), L in zip(axes, arms.values(), "ab"):
    m = df.FPR_perm_pct.mean()
    sns.histplot(df.FPR_perm_pct, color="red", ax=ax[0], bins=50, alpha=0.6)
    ax[0].axvline(m, color="red", lw=2, label=f"Observed mean ({m:.1f}%)")
    ax[0].axvline(5.0, color="black", ls="--", lw=2, label="Target type I error (5.0%)")
    ax[0].set_xlabel("Proportion of significant ROIs (%)", fontsize=FS)
    for a, raw, corr, lab in ((ax[1], "R2_Raw_Degree", "R2_Corr_Degree", "nodal degree"),
                              (ax[2], "R2_Raw_PC1", "R2_Corr_PC1", "connectome PC1")):
        sns.histplot(df[raw], color="gray", label="No label permutation", ax=a, alpha=0.5, bins=30)
        sns.histplot(df[corr], color="blue", label="Label permutation", ax=a, alpha=0.5, bins=30)
        a.set_xlabel(f"$R^2$ with {lab}", fontsize=FS)
    for a in ax:
        a.set_ylabel("Count", fontsize=FS); a.tick_params(labelsize=FS); a.legend(fontsize=FS - 6)
    ax[0].text(-0.02, 1.06, L, transform=ax[0].transAxes, fontsize=28, ha="right", va="bottom")
fig.tight_layout(); homogenize_fig(fig)
fig.savefig(FIG_DIR / "S16_null_simulation.png", dpi=300, bbox_inches="tight")